# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` as per best practices.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and discover all record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View main metadata attributes
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")
print(f"Published: {getattr(meta, 'datePublished', None)} | Version: {getattr(meta, 'version', None)}")
print(f"License: {meta.license}")

## 2. Data Overview
All data access is based on entities' `@id`. We'll enumerate the available record sets, and for each record set, list its fields and columns using the record set's `@id`.

In [ ]:
# List all record sets with their @id and human-readable name
record_sets = []
for rs in dataset.metadata.recordSets:
    print(f"@id: {rs['@id']} | name: {rs['name']}")
    record_sets.append(rs)

# For clarity, list available fields/columns for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    # List fields (high-level logical features)
    print("  Fields:")
    for field in rs['fields']:
        fid = field['@id']
        fname = field.get('name', '')
        ftype = field.get('dataType', '')
        print(f"    {fid} | {fname} | {ftype}")
    # List columns in source data
    print("  Columns:")
    for col in rs['columns']:
        cid = col['@id']
        cname = col.get('name', '')
        ctype = col.get('dataType', '')
        print(f"    {cid} | {cname} | {ctype}")

## 3. Data Extraction
We'll demonstrate loading the main record set (`@id` and field list from the previous cell) into a DataFrame for analysis.

First, define the record set and field `@id`s for use below.

In [ ]:
# Choose the main record set by its @id (from previous overview)
# Example: main data table is usually the first
main_record_set_id = record_sets[0]['@id']

# List all record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load each record set by its @id
for rs_id in record_set_ids:
    print(f"Loading records for {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns of the main record set
print("Columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by key fields (referenced by their `@id`), using the DataFrame loaded above.
- To filter, select a numeric field (`@id`).
- For normalization, operate on this numeric field.
- Group by a categorical field (`@id`) if available.

You can repeat this section for other fields or record sets as needed.

In [ ]:
# Identify candidate field @ids and map to DataFrame columns
df = dataframes[main_record_set_id]
# Example: assume there is a field/column like 'cr:Field/peptide_count' and 'cr:Field/sample_group' by @id (replace as per actual list)
# List DataFrame columns to assist in selecting field ids
print(df.columns.tolist())

# For usage, set the numeric field and group field by their @id (from schema overview or df columns above)
# E.g.,
# numeric_field_id = 'cr:Field/peptide_count'
# group_field_id = 'cr:Field/sample_group'
# Replace with actual df.columns value for peptide count and group, for illustration we guess below

numeric_field_id = [c for c in df.columns if ('count' in c or 'Count' in c)][0]  # fallback to first count field
group_candidates = [c for c in df.columns if 'group' in c.lower() or 'Group' in c]
group_field_id = group_candidates[0] if group_candidates else None

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If there's a group field, group and show mean
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric field and compare groups (if available).

*Note: You may need to install matplotlib if not present.*

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field_id].hist(bins=30, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.grid(False)
plt.show()

# If group field is available, boxplot by group
if group_field_id:
    plt.figure(figsize=(8,5))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated:
- Loading metadata and records from the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id`.
- Reviewing available record sets, fields, and columns dynamically.
- Loading and processing the dataset into DataFrames for analysis.
- Basic EDA steps: filtering, normalizing, grouping, and visualization.

You can customize this workflow for advanced data analysis and apply machine learning using the processed data from this Croissant-compliant dataset.